In [36]:
import requests
import datetime
import pandas as pd

In [37]:
def get_timestamps(filter, resolution, region = "DE", ):
    response = requests.get(f"https://smard.api.proxy.bund.dev/app/chart_data/{filter}/{region}/index_{resolution}.json")
    if response.status_code == 200:
        data = response.json()
        ## TODO: Check if data is valid
    return data

In [38]:
wasser_data = get_timestamps(1226, "quarterhour", "DE")

In [39]:
relevant_filter = [1225, 1226, 1228, 4066, 4067, 4068, 4070, 410, 4359, 4387, 4169, 3791, 123, 125, 715, 5097, 122]
filter_meaning = ["wind_off", "water", "else_renew", "biomass", "wind_on", "solar", "pump", "netzlast", "residual", "pump_verbr", "preis", "prog_off", "prog_on", "prog_solar", "prog_else", "prog_wind_solar", "prog_all" ]

df_filter = pd.DataFrame({"filter": relevant_filter, "meaning": filter_meaning})

In [40]:
len(relevant_filter)

17

In [41]:
wasser_data['timestamps']

[1419807600000,
 1420412400000,
 1421017200000,
 1421622000000,
 1422226800000,
 1422831600000,
 1423436400000,
 1424041200000,
 1424646000000,
 1425250800000,
 1425855600000,
 1426460400000,
 1427065200000,
 1427666400000,
 1428271200000,
 1428876000000,
 1429480800000,
 1430085600000,
 1430690400000,
 1431295200000,
 1431900000000,
 1432504800000,
 1433109600000,
 1433714400000,
 1434319200000,
 1434924000000,
 1435528800000,
 1436133600000,
 1436738400000,
 1437343200000,
 1437948000000,
 1438552800000,
 1439157600000,
 1439762400000,
 1440367200000,
 1440972000000,
 1441576800000,
 1442181600000,
 1442786400000,
 1443391200000,
 1443996000000,
 1444600800000,
 1445205600000,
 1445814000000,
 1446418800000,
 1447023600000,
 1447628400000,
 1448233200000,
 1448838000000,
 1449442800000,
 1450047600000,
 1450652400000,
 1451257200000,
 1451862000000,
 1452466800000,
 1453071600000,
 1453676400000,
 1454281200000,
 1454886000000,
 1455490800000,
 1456095600000,
 1456700400000,
 1457305

In [42]:
def create_data_table(filter, resolution, timestamps_list, region = "DE"):
    comp_list = []
    for timestamp in timestamps_list:
        response = requests.get(f"https://smard.api.proxy.bund.dev/app/chart_data/{filter}/{region}/{filter}_{region}_{resolution}_{timestamp}.json")
        if response.status_code == 200:
            data = response.json()
            comp_list.extend(data["series"])

    return comp_list

In [43]:
wasser_data["timestamps"][0:1]

[1419807600000]

In [44]:
df_filter.head()

,filter,meaning
0,1225,wind_off
1,1226,water
2,1228,else_renew
3,4066,biomass
4,4067,wind_on


In [45]:
water_series = create_data_table(1226, "quarterhour", wasser_data["timestamps"][0:2], "DE")

In [46]:
dt_json = {}
for i in range(len(df_filter)):
    filter = df_filter.iloc[i,0]
    timestamps = get_timestamps(filter, "quarterhour", "DE")
    meaning = df_filter.iloc[i,1]
    dt_json[meaning] = create_data_table(filter, "quarterhour", timestamps["timestamps"][-3:], "DE" )
    

In [47]:
def check_and_clean_data(data_dict):

    # for key, pairs in dt_json.items():
    #     dt_json[key] = [pair for pair in pairs if pair[1] is not None]

    for key, pairs in data_dict.items():
        dt_json[key] = sorted(
            [pair for pair in pairs if pair[1] is not None],
            key=lambda pair: pair[0]
        )

In [48]:
check_and_clean_data(dt_json)

In [49]:
def find_timestamp_gaps(dt_json, expected_gap_ms=15 * 60 * 1000):
    """
    Checks each key in dt_json for gaps between consecutive timestamps.

    Args:
        dt_json: dict like {
            "some_key": [[timestamp_ms, value], [timestamp_ms, value], ...],
            ...
        }
        expected_gap_ms: expected distance between timestamps in milliseconds.
                         Default = 15 minutes.

    Returns:
        dict where each key contains:
            - "has_gaps": True/False
            - "gaps": list of gap details
    """
    result = {}

    for key, pairs in dt_json.items():
        gaps = []

        for i in range(1, len(pairs)):
            prev_ts = pairs[i - 1][0]
            curr_ts = pairs[i][0]
            diff = curr_ts - prev_ts

            if diff != expected_gap_ms:
                gaps.append({
                    "prev_timestamp": prev_ts,
                    "current_timestamp": curr_ts,
                    "difference_ms": diff,
                    "missing_intervals": max(0, diff // expected_gap_ms - 1)
                })

        result[key] = {
            "has_gaps": len(gaps) > 0,
            "gaps": gaps
        }

    return result

In [50]:
gap_info = find_timestamp_gaps(dt_json)

for key, info in gap_info.items():
    if info["has_gaps"]:
        print(f"{key} has gaps:")
        for gap in info["gaps"]:
            print(gap)
    else:
        print(f"{key} has no gaps")

wind_off has gaps:
{'prev_timestamp': 1773294300000, 'current_timestamp': 1773296100000, 'difference_ms': 1800000, 'missing_intervals': 1}
water has gaps:
{'prev_timestamp': 1773294300000, 'current_timestamp': 1773296100000, 'difference_ms': 1800000, 'missing_intervals': 1}
else_renew has gaps:
{'prev_timestamp': 1772566200000, 'current_timestamp': 1772568000000, 'difference_ms': 1800000, 'missing_intervals': 1}
{'prev_timestamp': 1773294300000, 'current_timestamp': 1773296100000, 'difference_ms': 1800000, 'missing_intervals': 1}
biomass has gaps:
{'prev_timestamp': 1773294300000, 'current_timestamp': 1773296100000, 'difference_ms': 1800000, 'missing_intervals': 1}
wind_on has no gaps
solar has no gaps
pump has no gaps
netzlast has no gaps
residual has gaps:
{'prev_timestamp': 1773294300000, 'current_timestamp': 1773296100000, 'difference_ms': 1800000, 'missing_intervals': 1}
pump_verbr has no gaps
preis has no gaps
prog_off has no gaps
prog_on has no gaps
prog_solar has no gaps
prog_e

In [51]:
type(dt_json)

dict

In [ ]:
import pandas as pd

def dt_json_to_dataframe(dt_json, timestamp_unit="ms", convert_timestamp=True):
    """
    Convert a dt_json dictionary into one wide DataFrame.

    Parameters
    ----------
    dt_json : dict
        Example:
        {
            "wind_on": [[timestamp, value], [timestamp, value], ...],
            "solar": [[timestamp, value], [timestamp, value], ...]
        }
    timestamp_unit : str
        Time unit for pd.to_datetime when convert_timestamp=True.
    convert_timestamp : bool
        If True, convert timestamps to pandas datetime.

    Returns
    -------
    pd.DataFrame
        A DataFrame with one 'timestamp' column and one column per
        dt_json key, which matches your filter_meaning names.
    """
    series_frames = []

    for series_name, pairs in dt_json.items():
        if not pairs:
            series_frames.append(pd.DataFrame(columns=["timestamp", series_name]))
            continue

        series_df = pd.DataFrame(pairs, columns=["timestamp", series_name])
        series_df = series_df.drop_duplicates(subset=["timestamp"], keep="first")
        series_frames.append(series_df)

    if not series_frames:
        return pd.DataFrame(columns=["timestamp"])

    result_df = series_frames[0]

    for series_df in series_frames[1:]:
        result_df = result_df.merge(series_df, on="timestamp", how="outer")

    if convert_timestamp and not result_df.empty:
        result_df["timestamp"] = pd.to_datetime(result_df["timestamp"], unit=timestamp_unit, utc=True)

    ordered_columns = ["timestamp", *dt_json.keys()]
    result_df = result_df.reindex(columns=ordered_columns)
    result_df = result_df.sort_values("timestamp").reset_index(drop=True)

    return result_df


In [52]:
import pandas as pd

def analyze_dt_json_coverage(dt_json, freq="15min", timestamp_unit="ms"):
    """
    Analysiert für jede Serie in dt_json:
    - Start/Ende
    - Anzahl Punkte
    - Duplikate
    - erwartete Anzahl Punkte im Spannungsbereich
    - fehlende Zeitpunkte
    - zusammenhängende Lücken
    
    Parameters
    ----------
    dt_json : dict
        Beispiel:
        {
            "wind_on": [[timestamp, value], [timestamp, value], ...],
            "preis": [[timestamp, value], [timestamp, value], ...]
        }
    freq : str
        Erwartete Frequenz, z.B. "15min" oder "1h"
    timestamp_unit : str
        z.B. "ms" für Millisekunden
        
    Returns
    -------
    summary_df : pd.DataFrame
        Überblick pro Serie
    missing_dict : dict
        Fehlende einzelne Zeitpunkte pro Serie
    gaps_dict : dict
        Zusammenhängende Lückenintervalle pro Serie
    normalized_dict : dict
        Normalisierte DataFrames pro Serie
    """
    summary_rows = []
    missing_dict = {}
    gaps_dict = {}
    normalized_dict = {}

    expected_delta = pd.Timedelta(freq)

    for name, pairs in dt_json.items():
        # In DataFrame umwandeln
        df = pd.DataFrame(pairs, columns=["timestamp", "value"])

        # Falls leer
        if df.empty:
            summary_rows.append({
                "series": name,
                "n_rows_raw": 0,
                "n_rows_after_clean": 0,
                "start": pd.NaT,
                "end": pd.NaT,
                "duplicates": 0,
                "expected_points": 0,
                "missing_points": 0,
                "coverage_ratio": 0.0,
                "largest_gap_steps": None,
                "largest_gap_duration": None
            })
            missing_dict[name] = pd.DatetimeIndex([])
            gaps_dict[name] = pd.DataFrame(columns=["gap_start", "gap_end", "missing_steps", "duration"])
            normalized_dict[name] = df
            continue

        # timestamp -> datetime
        df["timestamp"] = pd.to_datetime(df["timestamp"], unit=timestamp_unit, utc=True)
        df = df.sort_values("timestamp").reset_index(drop=True)

        n_rows_raw = len(df)

        # None-Werte entfernen
        df = df[df["value"].notna()].copy()

        # Duplikate zählen
        duplicates = df["timestamp"].duplicated().sum()

        # Duplikate entfernen
        df = df.drop_duplicates(subset=["timestamp"], keep="first").copy()
        df = df.sort_values("timestamp").reset_index(drop=True)

        n_rows_after_clean = len(df)

        if df.empty:
            summary_rows.append({
                "series": name,
                "n_rows_raw": n_rows_raw,
                "n_rows_after_clean": 0,
                "start": pd.NaT,
                "end": pd.NaT,
                "duplicates": duplicates,
                "expected_points": 0,
                "missing_points": 0,
                "coverage_ratio": 0.0,
                "largest_gap_steps": None,
                "largest_gap_duration": None
            })
            missing_dict[name] = pd.DatetimeIndex([])
            gaps_dict[name] = pd.DataFrame(columns=["gap_start", "gap_end", "missing_steps", "duration"])
            normalized_dict[name] = df
            continue

        start = df["timestamp"].min()
        end = df["timestamp"].max()

        # Vollständiger erwarteter Index innerhalb des Serien-Zeitraums
        full_index = pd.date_range(start=start, end=end, freq=freq, tz="UTC")

        actual_index = pd.DatetimeIndex(df["timestamp"])
        missing = full_index.difference(actual_index)

        expected_points = len(full_index)
        missing_points = len(missing)
        coverage_ratio = len(actual_index) / expected_points if expected_points > 0 else 0.0

        # Zusammenhängende Lücken bestimmen
        gaps = []
        if len(missing) > 0:
            gap_start = missing[0]
            prev = missing[0]

            for ts in missing[1:]:
                if ts - prev == expected_delta:
                    prev = ts
                else:
                    missing_steps = int((prev - gap_start) / expected_delta) + 1
                    gaps.append({
                        "gap_start": gap_start,
                        "gap_end": prev,
                        "missing_steps": missing_steps,
                        "duration": prev - gap_start + expected_delta
                    })
                    gap_start = ts
                    prev = ts

            missing_steps = int((prev - gap_start) / expected_delta) + 1
            gaps.append({
                "gap_start": gap_start,
                "gap_end": prev,
                "missing_steps": missing_steps,
                "duration": prev - gap_start + expected_delta
            })

        gaps_df = pd.DataFrame(gaps)

        if not gaps_df.empty:
            largest_gap_row = gaps_df.sort_values("missing_steps", ascending=False).iloc[0]
            largest_gap_steps = int(largest_gap_row["missing_steps"])
            largest_gap_duration = largest_gap_row["duration"]
        else:
            largest_gap_steps = 0
            largest_gap_duration = pd.Timedelta(0)

        summary_rows.append({
            "series": name,
            "n_rows_raw": n_rows_raw,
            "n_rows_after_clean": n_rows_after_clean,
            "start": start,
            "end": end,
            "duplicates": int(duplicates),
            "expected_points": int(expected_points),
            "missing_points": int(missing_points),
            "coverage_ratio": round(coverage_ratio, 4),
            "largest_gap_steps": largest_gap_steps,
            "largest_gap_duration": largest_gap_duration
        })

        missing_dict[name] = missing
        gaps_dict[name] = gaps_df
        normalized_dict[name] = df

    summary_df = pd.DataFrame(summary_rows).sort_values("series").reset_index(drop=True)
    return summary_df, missing_dict, gaps_dict, normalized_dict

In [53]:
summary_df, missing_dict, gaps_dict, normalized_dict = analyze_dt_json_coverage(
    dt_json,
    freq="15min",
    timestamp_unit="ms"
)

In [54]:
print(summary_df)

             series  n_rows_raw  n_rows_after_clean                     start  \
0           biomass        1796                1796 2026-02-22 23:00:00+00:00   
1        else_renew        1795                1795 2026-02-22 23:00:00+00:00   
2          netzlast        1796                1796 2026-02-22 23:00:00+00:00   
3             preis        1920                1920 2026-02-22 23:00:00+00:00   
4          prog_all        1824                1824 2026-02-22 23:00:00+00:00   
5         prog_else        1824                1824 2026-02-22 23:00:00+00:00   
6          prog_off        1824                1824 2026-02-22 23:00:00+00:00   
7           prog_on        1824                1824 2026-02-22 23:00:00+00:00   
8        prog_solar        1824                1824 2026-02-22 23:00:00+00:00   
9   prog_wind_solar        1824                1824 2026-02-22 23:00:00+00:00   
10             pump        1796                1796 2026-02-22 23:00:00+00:00   
11       pump_verbr        1

In [55]:
def overall_coverage_window(summary_df):
    overall_start = summary_df["start"].min()
    overall_end = summary_df["end"].max()

    common_start = summary_df["start"].max()
    common_end = summary_df["end"].min()

    return {
        "overall_start": overall_start,
        "overall_end": overall_end,
        "common_start": common_start,
        "common_end": common_end,
        "common_window_valid": common_start <= common_end
    }

coverage_info = overall_coverage_window(summary_df)
print(coverage_info)

{'overall_start': Timestamp('2026-02-22 23:00:00+0000', tz='UTC'), 'overall_end': Timestamp('2026-03-14 22:45:00+0000', tz='UTC'), 'common_start': Timestamp('2026-02-22 23:00:00+0000', tz='UTC'), 'common_end': Timestamp('2026-03-13 15:45:00+0000', tz='UTC'), 'common_window_valid': True}


In [17]:
water_series = [i for i in water_series if i[1] is not None]

In [18]:
water_series[0:5]

[[1420066800000, 288.25],
 [1420067700000, 287.75],
 [1420068600000, 292.75],
 [1420069500000, 289.5],
 [1420070400000, 295.25]]

In [12]:

df_water = pd.DataFrame({'timestamp': [i for i in water_series[0]], 'value': [i for i in water_series[1]], })

In [ ]:

data["timestamps"]
dt = datetime.datetime.fromtimestamp(data["timestamps"][2] / 1000.0)
print(dt)


2015-01-12 00:00:00


In [33]:
1629065700000
data["timestamps"]
dt = datetime.datetime.fromtimestamp(1629068400000 / 1000.0)
print(dt)

2021-08-16 01:00:00
